# IFEST DAC 2026 — V8B Guarded Semantic Rescue

This notebook implements the frozen **V8B** design only.

### Frozen competition logic
- Base: exact frozen **V5**.
- Proposer: exact frozen **V8A** (`V5=0`, `v4_rule=v2_fallback`, frozen V3 blend `>= 0.5817181386596663`).
- V8B may only **revert a subset of V8A's proposed `0→1` rescues back to V5's `0`**.
- No new rescue candidates, no threshold expansion, no manual test surgery, no changes to V2/V4 structural decisions, and no `V5-positive → 0` edits.

### Factual guardrail
- Reuses the V6B mDeBERTa span-support formulation and official-train counterfactual supervision.
- Atomic factual spans: numbers/units, polarity markers, acronyms/entities, predicate-like terms, and high-IDF lexical claims.
- Retrieval masks the tested span from the query and uses **80-word windows, stride 40, top-2**.
- Three-state policy is implemented conservatively: lack of evidence alone is not a revert; a revert requires a high-confidence contradiction score from a calibrated factual guard.
- Small fixed, strongly regularized logistic guard combines neural span risks with deterministic contradiction diagnostics. No hyperparameter search is performed.

### Validation contract
- **2 repeats × 3 component-safe outer folds** over the frozen exact title/content connected components.
- Every train V8A rescue receives two independent OOF guard decisions, one per repeat, from models that never saw its component.
- Test uses the six fold specialists; within each repeat a candidate must be flagged by at least **2/3** fold guards, and both repeats must agree.
- Primary metric: full-system Macro F1 versus frozen V8A.
- Promotion requires: aggregate gain `> +0.002`, at least 5/6 non-negative outer folds, at least 4/6 positive outer folds, revert precision `>=75%`, true-rescue preservation `>=90%`, and component-bootstrap `P(Δ>0)>=95%`.
- Failure is closed: no `submission_v8b.csv` is created unless all validation and integrity gates pass.

Training is intended for Kaggle **T4×2**. No external labeled data or hidden test labels are used.

In [1]:
# Dependencies, reproducibility, and frozen V8B configuration.
import os, sys, re, io, gc, json, math, time, gzip, base64, hashlib, random, zipfile, unicodedata, warnings, subprocess, importlib.util, difflib, shutil
from pathlib import Path
from collections import Counter
from contextlib import nullcontext

missing=[]
for pkg,pip_name in [
    ('transformers','transformers>=4.44'),
    ('sentencepiece','sentencepiece'),
    ('sklearn','scikit-learn'),
]:
    if importlib.util.find_spec(pkg) is None: missing.append(pip_name)
if missing:
    print('Installing missing dependencies:', missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import transformers

warnings.filterwarnings('ignore')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY','1')
pd.set_option('display.max_colwidth',180)

# ---------------- V8B FROZEN CONFIG ----------------
SEED=2026
REPEAT_SEEDS=(2026,260908)
N_REPEATS=2
N_FOLDS=3
BOOTSTRAP_REPS=1000
BOOTSTRAP_SEED=260985

# Frozen V8A proposer. DO NOT RETUNE.
V8_BLEND_MIN=0.5817181386596663
REQUIRED_V4_RULE='v2_fallback'

# Factual specialist: V6B lineage.
MODEL_NAME='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
MODEL_REVISION='b5113eb'
LOCAL_MODEL_PATH=os.environ.get('V8B_MODEL_PATH','').strip() or None
MAX_LENGTH=320
SPAN_WINDOW_WORDS=80
SPAN_WINDOW_STRIDE=40
SPAN_TOP_WINDOWS=2
MAX_SPANS=6
HIGH_IDF_QUANTILE=0.80
PAIR_SIMILARITY_MIN=0.80
MAX_CHANGED_SPAN_TOKENS=3
PAIR_MARGIN=0.50
STRONG_LOSS_WEIGHT=1.0
PAIR_LOSS_WEIGHT=0.50
ROW_MIL_WEIGHT=0.25

# T4x2 budget, intentionally matching the proven V6B training regime.
EPOCHS=2
ROW_BATCH_SIZE=4
EVAL_ROW_BATCH_SIZE=8
GRAD_ACCUM=6
LR=1.0e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.08
MAX_GRAD_NORM=1.0
USE_AMP=True
NUM_WORKERS=2

# Fixed factual guard calibrator. No search.
CALIBRATOR_C=0.10
CALIBRATOR_CLASS_WEIGHT='balanced'
MIN_REVERT_PRECISION=0.75
MIN_TRUE_RESCUE_PRESERVATION=0.90

# Promotion contract frozen from the V8B design.
MIN_AGG_GAIN=0.002
MIN_NONNEGATIVE_OUTER_FOLDS=5
MIN_POSITIVE_OUTER_FOLDS=4
MIN_BOOTSTRAP_POSITIVE=0.95
MIN_TEST_REVERTS=1
MAX_OBVIOUS_TEST_REVERT_FRACTION=0.50  # broad pathology guard only; candidate set is already capped at 18

EXPECTED_TRAIN_ROWS=14397
EXPECTED_TEST_ROWS=3603
EXPECTED_TRAIN_ID_SHA256='9abdabdd6cda9baac679b5cf004572e02326859fc2b13caa37af32c335e6c94a'
EXPECTED_TEST_ID_SHA256='e4763272ee9a832ad2141f979a48701222a6651550ba2a88cf6c89357094064c'
EXPECTED_V5_SUBMISSION_SHA256='698790750562fda1dcc7eea78ac428d079312254b535f9aedf85fb240422dfdc'
EXPECTED_V5_OOF_CM=np.array([[1242,195],[381,12579]])
EXPECTED_V5_OOF_MACRO=0.8946909330297752
EXPECTED_V8A_TRAIN_RESCUES=108
EXPECTED_V8A_TEST_RESCUES=18
EXPECTED_BASE_CANDIDATES_TRAIN=672
EXPECTED_BASE_CANDIDATES_TEST=110
EXPECTED_V5_TEST_NEGATIVES=379
RULE_NAMES=np.array(['v2_fallback','v2_graph_locked','test_content_canonical','test_title_canonical'],dtype=object)

OUTPUT_DIR=Path('/kaggle/working/ifest_v8b_output' if Path('/kaggle/working').exists() else '/mnt/data/ifest_v8b_output')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_COUNT=torch.cuda.device_count()
AMP_ENABLED=bool(USE_AMP and DEVICE.type=='cuda')
if DEVICE.type!='cuda':
    raise RuntimeError('V8B training requires Kaggle GPU. Select GPU T4 x2 before running.')
if GPU_COUNT>=2: print('T4x2 path enabled via torch.nn.DataParallel')
else: print('WARNING: fewer than 2 CUDA devices detected; methodology is unchanged but training will be slower.')
for i in range(GPU_COUNT):
    p=torch.cuda.get_device_properties(i); print(i,torch.cuda.get_device_name(i),f'VRAM={p.total_memory/1024**3:.1f} GB')
print(f'PyTorch={torch.__version__} | Transformers={transformers.__version__} | AMP={AMP_ENABLED}')

T4x2 path enabled via torch.nn.DataParallel
0 Tesla T4 VRAM=14.6 GB
1 Tesla T4 VRAM=14.6 GB
PyTorch=2.10.0+cu128 | Transformers=5.0.0 | AMP=True


In [2]:
# Frozen prior outputs: V5 OOF/base state, V3 blend scores, exact V7A DESIGN/LOCK split, and V5 test state.
_FROZEN_B64 = '''...'''
_FROZEN_SHA256 = '86a14bf3ecb631369a8ad0617a451fa86c3e926b96126a80616cb7e48a151eb7'
_blob = base64.b64decode(_FROZEN_B64.encode())
assert hashlib.sha256(_blob).hexdigest() == _FROZEN_SHA256
_npz = np.load(io.BytesIO(_blob), allow_pickle=False)
train_v5_label = _npz['train_v5_label'].astype(np.int8)
train_v4_rule_code = _npz['train_v4_rule'].astype(np.int8)
train_v5_rescue = _npz['train_v5_rescue'].astype(bool)
train_v3_blend = _npz['train_v3_blend'].astype(np.float64)
train_partition = _npz['train_partition'].astype(np.int8)  # 0=DESIGN, 1=LOCK
train_design_fold = _npz['train_design_fold'].astype(np.int8)
train_component = _npz['train_component'].astype(np.int32)
test_v5_label = _npz['test_v5_label'].astype(np.int8)
test_v4_rule_code = _npz['test_v4_rule'].astype(np.int8)
test_v5_rescue = _npz['test_v5_rescue'].astype(bool)
test_v3_blend = _npz['test_v3_blend'].astype(np.float64)
del _npz, _blob

assert len(train_v5_label) == EXPECTED_TRAIN_ROWS
assert len(test_v5_label) == EXPECTED_TEST_ROWS
assert np.isfinite(train_v3_blend).all() and np.isfinite(test_v3_blend).all()
assert ((train_v3_blend >= 0) & (train_v3_blend <= 1)).all()
assert ((test_v3_blend >= 0) & (test_v3_blend <= 1)).all()
train_v4_rule = RULE_NAMES[np.clip(train_v4_rule_code, 0, 3)]
test_v4_rule = RULE_NAMES[np.clip(test_v4_rule_code, 0, 3)]
print('Frozen artifact loaded:', _FROZEN_SHA256)

Frozen artifact loaded: 86a14bf3ecb631369a8ad0617a451fa86c3e926b96126a80616cb7e48a151eb7


In [3]:
# Load authoritative competition CSVs and verify exact alignment with the frozen V5/V8A arrays.
def find_competition_csv(filename, expected_cols, expected_rows):
    roots=[Path('/kaggle/input'),Path('/mnt/data/ifest_dac'),Path('/mnt/data/ifest_audit/penyisihan-ifest-2026-dac')]
    candidates=[]
    for root in roots:
        if root.exists(): candidates.extend(root.rglob(filename))
    good=[]
    for p in candidates:
        if '/kaggle/working/' in str(p): continue
        try:
            d=pd.read_csv(p,nrows=3)
            if d.columns.tolist()!=expected_cols: continue
            rows=len(pd.read_csv(p,usecols=[expected_cols[0]]))
            if rows==expected_rows: good.append(p)
        except Exception: pass
    if not good: raise FileNotFoundError(f'Could not find a valid {filename} under Kaggle Input')
    good=sorted(good,key=lambda p:('penyisihan-ifest-2026-dac' not in str(p).lower(),len(str(p))))
    return good[0]

def ids_sha256(series):
    return hashlib.sha256('\n'.join(series.astype(str).tolist()).encode()).hexdigest()

def macro_f1(y_true,y_pred):
    return float(f1_score(y_true,y_pred,average='macro',labels=[0,1],zero_division=0))

train_path=find_competition_csv('train.csv',['id','title','content','label'],EXPECTED_TRAIN_ROWS)
test_path=find_competition_csv('test.csv',['id','title','content'],EXPECTED_TEST_ROWS)
sample_path=find_competition_csv('sample_submission.csv',['id','label'],EXPECTED_TEST_ROWS)
train=pd.read_csv(train_path); test=pd.read_csv(test_path); sample=pd.read_csv(sample_path)
assert train.columns.tolist()==['id','title','content','label']
assert test.columns.tolist()==['id','title','content']
assert sample.columns.tolist()==['id','label']
assert sample.id.astype(str).tolist()==test.id.astype(str).tolist()
assert ids_sha256(train.id)==EXPECTED_TRAIN_ID_SHA256
assert ids_sha256(test.id)==EXPECTED_TEST_ID_SHA256

y=train.label.to_numpy(np.int8)
cm=confusion_matrix(y,train_v5_label,labels=[0,1])
assert np.array_equal(cm,EXPECTED_V5_OOF_CM),cm.tolist()
assert abs(macro_f1(y,train_v5_label)-EXPECTED_V5_OOF_MACRO)<1e-12
assert int((test_v5_label==0).sum())==EXPECTED_V5_TEST_NEGATIVES
v5_submission=sample[['id']].copy(); v5_submission['label']=test_v5_label.astype(int)
v5_sha=hashlib.sha256(v5_submission.to_csv(index=False).encode()).hexdigest()
assert v5_sha==EXPECTED_V5_SUBMISSION_SHA256

# Exact frozen V8A proposer; V8B may only reject some of these proposals.
def apply_v8a_rescue(v5_labels,v4_rule,v3_blend):
    base=np.asarray(v5_labels,np.int8); rules=np.asarray(v4_rule,dtype=object); blend=np.asarray(v3_blend,float)
    out=base.copy(); rescue=(base==0)&(rules==REQUIRED_V4_RULE)&(blend>=V8_BLEND_MIN); out[rescue]=1
    assert not np.any((base==1)&(out==0))
    assert np.all(rules[rescue]==REQUIRED_V4_RULE)
    return out,rescue

train_v8a_label,train_v8a_rescue=apply_v8a_rescue(train_v5_label,train_v4_rule,train_v3_blend)
test_v8a_label,test_v8a_rescue=apply_v8a_rescue(test_v5_label,test_v4_rule,test_v3_blend)
train_base_candidate=(train_v5_label==0)&(train_v4_rule==REQUIRED_V4_RULE)
test_base_candidate=(test_v5_label==0)&(test_v4_rule==REQUIRED_V4_RULE)
assert int(train_base_candidate.sum())==EXPECTED_BASE_CANDIDATES_TRAIN
assert int(test_base_candidate.sum())==EXPECTED_BASE_CANDIDATES_TEST
assert int(train_v8a_rescue.sum())==EXPECTED_V8A_TRAIN_RESCUES
assert int(test_v8a_rescue.sum())==EXPECTED_V8A_TEST_RESCUES
assert not np.any(train_v8a_rescue & train_v5_rescue)
assert not np.any(test_v8a_rescue & test_v5_rescue)
print('Dataset/frozen parity: OK')
print('V8A rescue candidates train/test:',int(train_v8a_rescue.sum()),int(test_v8a_rescue.sum()))
print('V8A OOF Macro F1:',f'{macro_f1(y,train_v8a_label):.9f}')

Dataset/frozen parity: OK
V8A rescue candidates train/test: 108 18
V8A OOF Macro F1: 0.907147280


## Text normalization, atomic factual spans, and masked evidence retrieval

Span extraction is deterministic and label-free. Missing evidence is retained as uncertainty, not automatically treated as contradiction. Explicit numeric/acronym/polarity conflict indicators are diagnostics fed to the fixed guard calibrator together with neural support risks.

In [4]:
TOKEN_RE=re.compile(r'[\w]+',flags=re.UNICODE)
SOURCE_SUFFIX_RE=re.compile(r'\s+kumparan\s+com\s*$',re.I)
STOPWORDS=set('yang dan di ke dari untuk pada dengan adalah itu ini dalam akan telah sudah juga atau sebagai karena agar saat oleh atas terhadap antara tentang setelah sebelum lebih paling bisa dapat masih para sebuah suatu jadi yakni yaitu ada ia dia mereka kami kita anda kamu nya pun per serta hingga kalau jika namun tetapi sementara ketika dimana mana apa siapa bagaimana mengapa'.split())
NEG_POLARITY=set('tidak tak belum bukan jangan tanpa gagal batal menolak ditolak dilarang larang negatif turun berkurang berhenti tutup'.split())
POS_POLARITY=set('berhasil mampu boleh setuju positif naik meningkat lanjut buka dibuka'.split())
POLARITY=NEG_POLARITY|POS_POLARITY
NUMBER_UNITS=set('persen % ribu juta miliar triliun orang kasus pasien anak hari pekan minggu bulan tahun jam menit detik dosis unit kali km kg ton rp usd dolar'.split())
ACRONYM_RE=re.compile(r'\b[A-Z][A-Z0-9]{1,6}\b')
NUMBER_UNIT_PATTERN=r'%|persen|ribu|juta|miliar|triliun|orang|kasus|pasien|hari|pekan|minggu|bulan|tahun|jam|menit|dosis|unit|kali|km|kg|ton|usd|dolar'
# Competition text has punctuation stripped, so decimals commonly appear as `268 3` instead of `268,3`.
# Parse those forms as one factual number when a semantic unit follows.
UNIT_NUMERIC_RE=re.compile(rf'(?<!\w)(Rp\s*)?(\d+)(?:(?:[\.,]\s*|\s+)(\d{{1,2}}))?\s*({NUMBER_UNIT_PATTERN})(?!\w)',re.I)
CURRENCY_NUMERIC_RE=re.compile(r'(?<!\w)(Rp\s*)(\d+)(?:(?:[\.,]\s*|\s+)(\d{1,2}))?(?!\w)',re.I)
BARE_NUMERIC_RE=re.compile(r'(?<!\w)\d+(?:[\.,]\d+)*(?!\w)')
PREDICATE_RE=re.compile(r'^(?:me|mem|men|meng|meny|ber|ter|di)[a-z]{3,}$',re.I)

def clean_ws(text):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(text)).replace('\xa0',' ')).strip()
def clean_title_for_model(text): return SOURCE_SUFFIX_RE.sub('',clean_ws(text)).strip()
def normalize_key(text): return ' '.join(TOKEN_RE.findall(clean_ws(text).lower()))
def tokens(text): return TOKEN_RE.findall(normalize_key(text))
def model_words(text): return TOKEN_RE.findall(clean_title_for_model(text))

def number_signatures(text):
    text=str(text); out=[]; occupied=[]
    def overlaps(span): return any(not (span[1]<=a or span[0]>=b) for a,b in occupied)
    # Unit-bearing numbers are the safest comparison anchors and also reconstruct stripped decimals.
    for m in UNIT_NUMERIC_RE.finditer(text):
        whole=m.group(2) or ''; frac=m.group(3) or ''; unit=normalize_key(m.group(4) or '')
        digits=re.sub(r'\D','',whole+frac)
        if digits:
            out.append((digits,unit,m.group(0).strip())); occupied.append(m.span())
    # Currency values without an explicit unit retain `rp` as a conservative comparison unit.
    for m in CURRENCY_NUMERIC_RE.finditer(text):
        if overlaps(m.span()): continue
        whole=m.group(2) or ''; frac=m.group(3) or ''; digits=re.sub(r'\D','',whole+frac)
        if digits:
            out.append((digits,'rp',m.group(0).strip())); occupied.append(m.span())
    # Bare numbers remain useful neural spans, but are not used to assert an alternative-number conflict.
    for m in BARE_NUMERIC_RE.finditer(text):
        if overlaps(m.span()): continue
        digits=re.sub(r'\D','',m.group(0))
        if digits: out.append((digits,'',m.group(0).strip()))
    return out

def acronym_tokens(text): return [x.upper() for x in ACRONYM_RE.findall(str(text))]

for df in (train,test):
    df['title_model']=df.title.map(clean_title_for_model)
    df['title_key']=df.title_model.map(normalize_key)
    df['content_key']=df.content.map(normalize_key)
    df['pair_key']=df.title_key+'|||'+df.content_key

# Label-free IDF, same corpus convention as V6B.
doc_freq=Counter(); N_DOCS=len(train)
for t,c in tqdm(zip(train.title_model,train.content),total=N_DOCS,desc='IDF corpus'):
    doc_freq.update(set(tokens(t))|set(tokens(c)))
idf={tok:math.log((N_DOCS+1)/(df+1))+1.0 for tok,df in doc_freq.items()}
IDF_FLOOR=math.log((N_DOCS+1)/2)+1.0
HIGH_IDF_THRESHOLD=float(np.quantile(np.asarray(list(idf.values()),float),HIGH_IDF_QUANTILE))
def tok_idf(tok): return idf.get(tok,IDF_FLOOR)

def make_windows(content,window_words=SPAN_WINDOW_WORDS,stride=SPAN_WINDOW_STRIDE):
    words=clean_ws(content).split()
    if not words: return [(0,'')]
    if len(words)<=window_words: return [(0,' '.join(words))]
    starts=list(range(0,max(1,len(words)-window_words+1),stride)); last=max(0,len(words)-window_words)
    if starts[-1]!=last: starts.append(last)
    return [(s,' '.join(words[s:s+window_words])) for s in starts]

def title_window_score(title_toks,window_text):
    tset=set(title_toks)
    if not tset: return 0.0
    wset=set(tokens(window_text)); denom=sum(tok_idf(t) for t in tset)+1e-9
    return float(sum(tok_idf(t) for t in tset if t in wset)/denom)

def remove_target_from_title(title,target_tokens):
    ws=model_words(title); low=[normalize_key(x) for x in ws]; target=[normalize_key(x) for x in target_tokens if normalize_key(x)]
    start=-1
    if target:
        for i in range(len(low)-len(target)+1):
            if low[i:i+len(target)]==target: start=i; break
    kept=ws if start<0 else ws[:start]+ws[start+len(target):]
    return ' '.join(kept).strip() or 'berita ini'

def span_priority(kind): return {'number':6,'polarity':6,'acronym':6,'entity':4,'predicate':3,'high_idf':2,'fallback':0}.get(kind,0)

def extract_candidate_spans(title):
    ws=model_words(title); low=[normalize_key(w) for w in ws]; cand={}
    def add_text(txt,kind,score):
        txt=clean_ws(txt); key=normalize_key(txt)
        if not key: return
        rec={'text':txt,'norm':key,'kind':kind,'score':float(score),'tokens':key.split()}
        old=cand.get((key,kind))
        if old is None or score>old['score']: cand[(key,kind)]=rec
    # Full numeric expressions preserve decimals/units better than token-only extraction.
    for digits,unit,raw in number_signatures(title): add_text(raw,'number',200.0+len(digits))
    # Acronyms capture compact organization/entity substitutions such as MA/MK, BPOM, MUI, FPI.
    for ac in acronym_tokens(title): add_text(ac,'acronym',180.0+tok_idf(normalize_key(ac)))
    for i,w in enumerate(low):
        if w in POLARITY:
            txt=' '.join(ws[i:min(len(ws),i+2)]); add_text(txt,'polarity',170.0+sum(tok_idf(x) for x in tokens(txt)))
        if w and w not in STOPWORDS and PREDICATE_RE.match(w) and tok_idf(w)>=HIGH_IDF_THRESHOLD:
            add_text(ws[i],'predicate',60.0+tok_idf(w))
    # Title-case multi-token spans provide candidate person/org/location bindings without an external NER model.
    # Numeric/polarity/acronym slots still outrank them, so they cannot crowd out the highest-confidence factual slots.
    for n in (2,3):
        for i in range(0,len(ws)-n+1):
            seg=ws[i:i+n]; norms=[normalize_key(x) for x in seg]
            if any((not x) or x in STOPWORDS for x in norms): continue
            if all(x[:1].isupper() for x in seg):
                add_text(' '.join(seg),'entity',90.0+sum(tok_idf(x) for x in norms)+0.05*n)
    for n in (1,2,3):
        for i in range(0,len(ws)-n+1):
            seg=low[i:i+n]
            if not any(seg) or all(x in STOPWORDS for x in seg): continue
            vals=[tok_idf(x) for x in seg if x]
            if vals and max(vals)>=HIGH_IDF_THRESHOLD:
                add_text(' '.join(ws[i:i+n]),'high_idf',sum(vals)+0.05*n)
    if not cand and ws:
        scores=[tok_idf(normalize_key(w)) for w in ws]; i=int(np.argmax(scores)); add_text(ws[i],'fallback',scores[i])
    vals=list(cand.values())
    vals.sort(key=lambda r:(span_priority(r['kind']),r['score'],len(r['tokens']),r['norm']),reverse=True)
    return vals[:MAX_SPANS]

def explicit_span_diagnostics(target_span,kind,evidence):
    ev=' '.join(evidence); ev_key=normalize_key(ev); target_key=normalize_key(target_span)
    target_present=int(bool(target_key and target_key in ev_key))
    num_alt=acr_alt=pol_alt=0
    if kind=='number':
        ts=number_signatures(target_span); es=number_signatures(ev)
        if ts:
            td,tu,_=ts[0]
            exact=any(ed==td and (not tu or eu==tu) for ed,eu,_ in es)
            # Alternative-number contradiction is intentionally conservative: require the same semantic unit.
            # Bare years/counts without a unit can be numerous in an article and must not trigger a hard conflict.
            alt=any(ed!=td and bool(tu) and eu==tu for ed,eu,_ in es)
            target_present=int(exact); num_alt=int((not exact) and alt)
    elif kind=='acronym':
        ta=set(acronym_tokens(target_span)); ea=set(acronym_tokens(ev))
        exact=bool(ta & ea); target_present=int(exact); acr_alt=int((not exact) and bool(ea-ta))
    elif kind=='polarity':
        tt=set(tokens(target_span)); ee=set(tokens(ev))
        target_neg=bool(tt&NEG_POLARITY); target_pos=bool(tt&POS_POLARITY)
        ev_neg=bool(ee&NEG_POLARITY); ev_pos=bool(ee&POS_POLARITY)
        pol_alt=int((target_neg and ev_pos and not ev_neg) or (target_pos and ev_neg and not ev_pos))
    return target_present,num_alt,acr_alt,pol_alt

def retrieve_for_span(title,target_span,content,kind):
    context=remove_target_from_title(title,tokens(target_span)); query_toks=tokens(context)
    windows=make_windows(content); scored=[]
    for wi,(_,txt) in enumerate(windows): scored.append((title_window_score(query_toks,txt),-wi,wi))
    scored.sort(reverse=True); chosen=[x[2] for x in scored[:SPAN_TOP_WINDOWS]] or [0]
    evidence=[windows[i][1] for i in chosen]
    while len(evidence)<SPAN_TOP_WINDOWS: evidence.append(evidence[-1])
    scores=[float(scored[j][0]) for j in range(min(SPAN_TOP_WINDOWS,len(scored)))]
    while len(scores)<SPAN_TOP_WINDOWS: scores.append(scores[-1] if scores else 0.0)
    claim=f'Fakta yang diuji: {target_span}. Konteks judul tanpa fakta ini: {context}.'
    present,nalt,aalt,palt=explicit_span_diagnostics(target_span,kind,evidence)
    return {'span':target_span,'span_norm':normalize_key(target_span),'kind':kind,'context':context,'claim':claim,'evidence':evidence,
            'retrieval_scores':scores,'target_present':present,'number_alt_conflict':nalt,'acronym_alt_conflict':aalt,'polarity_alt_conflict':palt}

def build_row_span_bank(df,indices,desc):
    bank={}
    for idx in tqdm(list(map(int,indices)),desc=desc):
        r=df.iloc[idx]; specs=[]
        for s in extract_candidate_spans(r.title_model):
            rec=retrieve_for_span(r.title_model,s['text'],r.content,s['kind']); rec['rank_score']=s['score']; specs.append(rec)
        if not specs: raise RuntimeError(f'No factual span candidate for id={r.id}')
        bank[idx]=specs
    return bank

IDF corpus:   0%|          | 0/14397 [00:00<?, ?it/s]

## Component-safe repeated outer folds and official-train supervision

The 601-style localized counterfactual mining rule is unchanged from V6B: same-content canonical positive versus corrupted negative, title similarity `>=0.80`, exactly one edit operation, and changed side length `<=3` tokens. No synthetic negative is generated.

In [5]:
# Build two independent 3-fold component-safe split registries.
repeat_fold=np.full((N_REPEATS,len(train)),-1,dtype=np.int8)
repeat_splits=[]
for r,rs in enumerate(REPEAT_SEEDS):
    sgkf=StratifiedGroupKFold(n_splits=N_FOLDS,shuffle=True,random_state=rs)
    splits=[]
    for f,(tr_idx,va_idx) in enumerate(sgkf.split(np.zeros(len(train)),y,groups=train_component)):
        tr_idx=np.asarray(tr_idx,int); va_idx=np.asarray(va_idx,int)
        assert set(train_component[tr_idx]).isdisjoint(set(train_component[va_idx]))
        repeat_fold[r,va_idx]=f; splits.append((tr_idx,va_idx))
    assert (repeat_fold[r]>=0).all()
    repeat_splits.append(splits)
print('Repeated component-safe outer folds:',[[len(v) for _,v in s] for s in repeat_splits])

# Exact V6B counterfactual-pair mining.
def align_counterfactual_titles(pos_title,neg_title):
    a=model_words(pos_title); b=model_words(neg_title); al=[x.lower() for x in a]; bl=[x.lower() for x in b]
    sm=difflib.SequenceMatcher(a=al,b=bl,autojunk=False); sim=float(sm.ratio())
    edits=[op for op in sm.get_opcodes() if op[0]!='equal']
    if sim<PAIR_SIMILARITY_MIN or len(edits)!=1: return None
    tag,i1,i2,j1,j2=edits[0]; apos=a[i1:i2]; bneg=b[j1:j2]
    if max(len(apos),len(bneg))==0: return None
    if len(apos)>MAX_CHANGED_SPAN_TOKENS or len(bneg)>MAX_CHANGED_SPAN_TOKENS: return None
    return {'similarity':sim,'edit':tag,'pos_span':' '.join(apos),'neg_span':' '.join(bneg),'pos_len':len(apos),'neg_len':len(bneg)}

def mine_clean_counterfactual_pairs(df):
    rows=[]
    for ckey,g in df.groupby('content_key',sort=False):
        pos=g[g.label==1]; neg=g[g.label==0]; pos_keys=pos.title_key.unique().tolist()
        if len(pos_keys)!=1 or len(neg)==0: continue
        pg=pos[pos.title_key==pos_keys[0]]; pidx=int(pg.index[0]); ptitle=str(pg.iloc[0].title_model)
        for nkey,ng in neg.groupby('title_key',sort=False):
            nidx=int(ng.index[0]); ntitle=str(ng.iloc[0].title_model); info=align_counterfactual_titles(ptitle,ntitle)
            if info is None: continue
            assert int(train_component[pidx])==int(train_component[nidx])
            rows.append({'pair_id':len(rows),'content_key':ckey,'component':int(train_component[pidx]),'pos_idx':pidx,'neg_idx':nidx,'pos_title':ptitle,'neg_title':ntitle,**info})
    return pd.DataFrame(rows)

clean_pairs=mine_clean_counterfactual_pairs(train)
if len(clean_pairs)==0: raise RuntimeError('No clean counterfactual pairs mined; normalization drifted.')
print('Clean counterfactual pairs:',len(clean_pairs))
if len(clean_pairs)!=601: print('WARNING: historical V6B mined 601 pairs; current deterministic extraction produced',len(clean_pairs))

# Weak supervision remains the official graph-unresolved/pure semantic-fallback population.
weak_mask=(train_v4_rule==REQUIRED_V4_RULE)
weak_indices=np.flatnonzero(weak_mask)
train_rescue_idx=np.flatnonzero(train_v8a_rescue)
test_rescue_idx=np.flatnonzero(test_v8a_rescue)
needed_train_idx=np.unique(np.concatenate([weak_indices,clean_pairs.pos_idx.to_numpy(int),clean_pairs.neg_idx.to_numpy(int),train_rescue_idx]))
train_span_bank=build_row_span_bank(train,needed_train_idx,'V8B factual span bank train')
test_span_bank=build_row_span_bank(test,test_rescue_idx,'V8B factual span bank test')
print('Weak rows:',len(weak_indices),'| V8A train/test rescues:',len(train_rescue_idx),len(test_rescue_idx))

Repeated component-safe outer folds: [[4798, 4807, 4792], [4765, 4783, 4849]]
Clean counterfactual pairs: 601


V8B factual span bank train:   0%|          | 0/13563 [00:00<?, ?it/s]

V8B factual span bank test:   0%|          | 0/18 [00:00<?, ?it/s]

Weak rows: 12384 | V8A train/test rescues: 108 18


## Binary span-support verifier

The encoder uses FP32 master weights. AMP is restricted to the transformer forward pass; BCE/MIL/ranking arithmetic stays FP32, preserving the V6B numerical fix that avoids CUDA autocast BCE failures.

In [6]:
def materialize_pair(row):
    content=str(train.loc[int(row.pos_idx),'content']); items=[]; pos_item=neg_item=-1
    if str(row.pos_span).strip():
        v=retrieve_for_span(row.pos_title,row.pos_span,content,'high_idf'); v['label']=1; v['side']='positive'; pos_item=len(items); items.append(v)
    if str(row.neg_span).strip():
        v=retrieve_for_span(row.neg_title,row.neg_span,content,'high_idf'); v['label']=0; v['side']='negative'; neg_item=len(items); items.append(v)
    return {'pair_id':int(row.pair_id),'component':int(row.component),'edit':row.edit,'items':items,'pos_item':pos_item,'neg_item':neg_item,
            'pos_span':row.pos_span,'neg_span':row.neg_span,'pos_idx':int(row.pos_idx),'neg_idx':int(row.neg_idx)}
strong_pairs=[materialize_pair(r) for r in clean_pairs.itertuples()]

def resolve_model_source():
    if LOCAL_MODEL_PATH: return LOCAL_MODEL_PATH
    root=Path('/kaggle/input')
    if root.exists():
        for cfg in root.rglob('config.json'):
            p=cfg.parent; name=str(p).lower()
            if 'mdeberta' in name and (p/'tokenizer_config.json').exists():
                print('Using local Kaggle mDeBERTa model:',p); return str(p)
    return MODEL_NAME

def model_kwargs(source): return {} if LOCAL_MODEL_PATH or source!=MODEL_NAME else {'revision':MODEL_REVISION}
def ensure_fp32_master_weights(model,verbose=False):
    if any(p.is_floating_point() and p.dtype!=torch.float32 for p in model.parameters()): model.float()
    rem={str(p.dtype) for p in model.parameters() if p.is_floating_point() and p.dtype!=torch.float32}
    if rem: raise RuntimeError(f'FP32 master-weight invariant failed: {sorted(rem)}')
    if verbose: print('Master parameter dtypes:',sorted({str(p.dtype) for p in model.parameters() if p.is_floating_point()}))
    return model

def load_binary_span_model():
    source=resolve_model_source(); kw=model_kwargs(source)
    tok=AutoTokenizer.from_pretrained(source,use_fast=True,**kw)
    try: model=AutoModelForSequenceClassification.from_pretrained(source,num_labels=2,ignore_mismatched_sizes=True,dtype=torch.float32,**kw)
    except TypeError: model=AutoModelForSequenceClassification.from_pretrained(source,num_labels=2,ignore_mismatched_sizes=True,torch_dtype=torch.float32,**kw)
    model=ensure_fp32_master_weights(model); model.config.id2label={0:'UNSUPPORTED',1:'SUPPORTED'}; model.config.label2id={'UNSUPPORTED':0,'SUPPORTED':1}; model.config.use_cache=False
    return tok,model,source

def unwrap_model(model): return model.module if isinstance(model,nn.DataParallel) else model
def amp_context(): return torch.autocast('cuda',dtype=torch.float16) if AMP_ENABLED else nullcontext()
def make_grad_scaler():
    try: return torch.amp.GradScaler('cuda',enabled=AMP_ENABLED)
    except (AttributeError,TypeError): return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
def loader_kwargs(): return {'num_workers':NUM_WORKERS,'pin_memory':DEVICE.type=='cuda','persistent_workers':NUM_WORKERS>0}
def encode_views(tokenizer,evidences,claims): return tokenizer(evidences,claims,padding=True,truncation='only_first',max_length=MAX_LENGTH,return_tensors='pt')
def to_device(enc): return {k:v.to(DEVICE,non_blocking=True) for k,v in enc.items()}
def support_probs_from_logits(logits): return torch.softmax(logits.float(),dim=-1)[:,1]

class IndexDataset(Dataset):
    def __init__(self,indices): self.indices=np.asarray(indices,dtype=np.int64)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i): return int(self.indices[i])
def collate_indices(xs): return list(map(int,xs))

def flatten_rows(row_indices,bank):
    evid=[]; claims=[]; row_groups=[]
    for idx in row_indices:
        entries=[]
        for spec in bank[idx]:
            pos=[]
            for ev in spec['evidence']:
                pos.append(len(evid)); evid.append(ev); claims.append(spec['claim'])
            entries.append(pos)
        row_groups.append(entries)
    return evid,claims,row_groups

def flatten_pair(pair):
    evid=[]; claims=[]; groups=[]; labels=[]
    for item in pair['items']:
        pos=[]
        for ev in item['evidence']:
            pos.append(len(evid)); evid.append(ev); claims.append(item['claim'])
        groups.append(pos); labels.append(int(item['label']))
    return evid,claims,groups,labels

def aggregate_group_max(seq_support,groups):
    return torch.stack([seq_support[torch.as_tensor(g,device=seq_support.device,dtype=torch.long)].max() for g in groups]) if groups else torch.empty(0,device=seq_support.device)

def weak_row_mil_loss(seq_support,row_groups,row_labels):
    losses=[]
    for groups,yv in zip(row_groups,row_labels):
        span_support=torch.stack([seq_support[torch.as_tensor(g,device=seq_support.device,dtype=torch.long)].max() for g in groups]).clamp(1e-6,1-1e-6)
        loss=(-torch.log(span_support)).mean() if int(yv)==1 else -torch.log1p(-span_support.min())
        losses.append(loss)
    return torch.stack(losses).mean()

def score_support_matrix(model,tokenizer,indices,bank,desc):
    out={}
    dl=DataLoader(IndexDataset(indices),batch_size=EVAL_ROW_BATCH_SIZE,shuffle=False,collate_fn=collate_indices,**loader_kwargs())
    model.eval()
    with torch.inference_mode():
        for rows in tqdm(dl,desc=desc):
            evid,claims,row_groups=flatten_rows(rows,bank); enc=to_device(encode_views(tokenizer,evid,claims))
            with amp_context(): logits=model(**enc,return_dict=False)[0]
            seq=support_probs_from_logits(logits).float()
            for idx,groups in zip(rows,row_groups):
                vals=[float(seq[torch.as_tensor(g,device=seq.device,dtype=torch.long)].max().item()) for g in groups]
                out[int(idx)]=np.asarray(vals,dtype=np.float32)
            del enc,logits,seq
    return out

def score_pairs(model,tokenizer,pairs,desc):
    rows=[]
    model.eval()
    with torch.inference_mode():
        for pair in tqdm(pairs,desc=desc):
            evid,claims,groups,labels=flatten_pair(pair)
            if not evid: continue
            enc=to_device(encode_views(tokenizer,evid,claims))
            with amp_context(): logits=model(**enc,return_dict=False)[0]
            seq=support_probs_from_logits(logits).float(); item=aggregate_group_max(seq,groups).detach().cpu().numpy()
            if pair['pos_item']>=0 and pair['neg_item']>=0:
                rows.append({'pair_id':pair['pair_id'],'component':pair['component'],'edit':pair['edit'],'pos_support':float(item[pair['pos_item']]),'neg_support':float(item[pair['neg_item']]),'correct':int(item[pair['pos_item']]>item[pair['neg_item']])})
            del enc,logits,seq
    return rows

## Train six component-safe specialists

Each outer model sees only reference components. It scores only its held-out V8A rescue candidates plus the 18 frozen test V8A proposals. Models are deleted immediately after scoring; checkpoints are not retained because V8B only needs OOF/test factual features.

In [7]:
train_support=np.full((N_REPEATS,len(train),MAX_SPANS),np.nan,dtype=np.float32)
test_support=np.full((N_REPEATS,N_FOLDS,len(test),MAX_SPANS),np.nan,dtype=np.float32)
training_history=[]; pair_metric_rows=[]; model_sources=[]

def train_one_specialist(rep,fold,tr_idx,va_idx):
    seed=REPEAT_SEEDS[rep]+100*fold; seed_everything(seed)
    tr_components=set(train_component[tr_idx]); va_components=set(train_component[va_idx])
    assert tr_components.isdisjoint(va_components)
    weak_idx=np.asarray([i for i in tr_idx if weak_mask[i]],dtype=np.int64)
    ref_pairs=[p for p in strong_pairs if p['component'] in tr_components]
    val_pairs=[p for p in strong_pairs if p['component'] in va_components]
    if not ref_pairs: raise RuntimeError(f'repeat={rep} fold={fold}: no strong reference pairs')

    tok,base_model,source=load_binary_span_model(); model_sources.append(source)
    base_model.to(DEVICE); ensure_fp32_master_weights(base_model)
    model=nn.DataParallel(base_model) if GPU_COUNT>=2 else base_model
    dtypes=sorted({str(p.dtype) for p in unwrap_model(model).parameters() if p.is_floating_point()})
    if dtypes!=['torch.float32']: raise RuntimeError(f'Unexpected master dtypes: {dtypes}')

    gen=torch.Generator(); gen.manual_seed(seed)
    loader=DataLoader(IndexDataset(weak_idx),batch_size=ROW_BATCH_SIZE,shuffle=True,generator=gen,collate_fn=collate_indices,**loader_kwargs())
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=make_grad_scaler(); updates=math.ceil(len(loader)/GRAD_ACCUM)*EPOCHS
    sched=get_linear_schedule_with_warmup(opt,int(updates*WARMUP_RATIO),updates); opt.zero_grad(set_to_none=True)

    for epoch in range(EPOCHS):
        model.train(); rng=np.random.default_rng(seed+1000*epoch); order=rng.permutation(len(ref_pairs)); ls=[]; lstrong=[]; lpair=[]; lweak=[]; t0=time.perf_counter()
        pbar=tqdm(loader,desc=f'V8B r{rep} f{fold} epoch {epoch+1}/{EPOCHS}')
        for step,rows in enumerate(pbar):
            we,wc,wgroups=flatten_rows(rows,train_span_bank)
            pair=ref_pairs[int(order[step%len(order)])]; se,sc,sgroups,slabels=flatten_pair(pair)
            enc=to_device(encode_views(tok,we+se,wc+sc))
            with amp_context(): logits=model(**enc,return_dict=False)[0]
            support=support_probs_from_logits(logits)
            weak_loss=weak_row_mil_loss(support[:len(we)],wgroups,y[rows])
            strong_loss=torch.zeros((),device=DEVICE,dtype=torch.float32); pair_loss=torch.zeros((),device=DEVICE,dtype=torch.float32)
            if se:
                item=aggregate_group_max(support[len(we):],sgroups).clamp(1e-6,1-1e-6)
                lab=torch.as_tensor(slabels,device=DEVICE,dtype=torch.float32)
                strong_loss=F.binary_cross_entropy(item,lab)
                if pair['pos_item']>=0 and pair['neg_item']>=0:
                    pair_loss=F.relu(PAIR_MARGIN-(item[pair['pos_item']]-item[pair['neg_item']]))
            loss=STRONG_LOSS_WEIGHT*strong_loss+PAIR_LOSS_WEIGHT*pair_loss+ROW_MIL_WEIGHT*weak_loss
            scaler.scale(loss/GRAD_ACCUM).backward()
            ls.append(float(loss.detach())); lstrong.append(float(strong_loss.detach())); lpair.append(float(pair_loss.detach())); lweak.append(float(weak_loss.detach()))
            if ((step+1)%GRAD_ACCUM==0) or (step+1==len(loader)):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
                bad=[p.grad.dtype for p in unwrap_model(model).parameters() if p.grad is not None and p.grad.dtype!=torch.float32]
                if bad: raise RuntimeError(f'Non-FP32 master gradients: {set(map(str,bad))}')
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if step%50==0: pbar.set_postfix(loss=f'{np.mean(ls[-50:]):.3f}',strong=f'{np.mean(lstrong[-50:]):.3f}',weak=f'{np.mean(lweak[-50:]):.3f}')
            del enc,logits,support,loss
        training_history.append({'repeat':rep,'fold':fold,'epoch':epoch+1,'loss':float(np.mean(ls)),'strong_loss':float(np.mean(lstrong)),'pair_loss':float(np.mean(lpair)),'weak_mil_loss':float(np.mean(lweak)),'seconds':time.perf_counter()-t0,'weak_rows':len(weak_idx),'strong_pairs':len(ref_pairs)})

    prows=score_pairs(model,tok,val_pairs,f'r{rep} f{fold} pair validation')
    for rr in prows: rr.update({'repeat':rep,'fold':fold})
    pair_metric_rows.extend(prows)

    va_cand=np.asarray([i for i in va_idx if train_v8a_rescue[i]],dtype=np.int64)
    scored=score_support_matrix(model,tok,va_cand,train_span_bank,f'r{rep} f{fold} OOF V8A rescues')
    for idx,arr in scored.items(): train_support[rep,idx,:len(arr)]=arr
    tscored=score_support_matrix(model,tok,test_rescue_idx,test_span_bank,f'r{rep} f{fold} test V8A rescues')
    for idx,arr in tscored.items(): test_support[rep,fold,idx,:len(arr)]=arr

    del model,base_model,tok,loader,opt,scaler,sched
    gc.collect(); torch.cuda.empty_cache()

for rep in range(N_REPEATS):
    for fold,(tr_idx,va_idx) in enumerate(repeat_splits[rep]):
        train_one_specialist(rep,fold,tr_idx,va_idx)

# Completeness guards: each train rescue has one held-out specialist score per repeat; every test rescue has all six.
for rep in range(N_REPEATS):
    for idx in train_rescue_idx:
        n=len(train_span_bank[int(idx)]); assert np.isfinite(train_support[rep,idx,:n]).all()
for rep in range(N_REPEATS):
    for fold in range(N_FOLDS):
        for idx in test_rescue_idx:
            n=len(test_span_bank[int(idx)]); assert np.isfinite(test_support[rep,fold,idx,:n]).all()
print('Six specialists completed; all required OOF/test support vectors are present.')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

V8B r1 f2 epoch 2/2:   0%|          | 0/2057 [00:00<?, ?it/s]

r1 f2 pair validation:   0%|          | 0/205 [00:00<?, ?it/s]

r1 f2 OOF V8A rescues:   0%|          | 0/5 [00:00<?, ?it/s]

r1 f2 test V8A rescues:   0%|          | 0/3 [00:00<?, ?it/s]

Six specialists completed; all required OOF/test support vectors are present.


## Fixed factual feature contract and cross-fitted guard calibration

The guard target is **false V8A rescue** (`train label = 0`) among the 108 frozen V8A proposals. The calibrator is fixed `StandardScaler + LogisticRegression(C=0.10, class_weight='balanced')`. For each outer fold, the threshold is selected only on the other two folds by full-system Macro F1, subject to `>=75%` revert precision and `>=90%` preservation of true V8A rescues. The held-out fold is then evaluated once.

In [8]:
FEATURE_NAMES=[
    'neural_max_risk','neural_mean_risk','number_risk','polarity_risk','entity_risk','predicate_risk',
    'number_alt_conflict','acronym_alt_conflict','polarity_alt_conflict','target_absent_ratio',
    'retrieval_confidence','retrieval_floor'
]

def feature_vector(specs,supports):
    supports=np.asarray(supports[:len(specs)],float)
    if len(supports)!=len(specs) or not np.isfinite(supports).all(): raise ValueError('Non-finite/misaligned support vector')
    risks=1.0-supports
    kinds=[s['kind'] for s in specs]
    def kind_risk(allowed):
        vals=[risks[i] for i,k in enumerate(kinds) if k in allowed]
        return float(max(vals)) if vals else 0.0
    retrieval=[float(s['retrieval_scores'][0]) for s in specs]
    present=[int(s['target_present']) for s in specs]
    return np.asarray([
        float(risks.max()),float(risks.mean()),kind_risk({'number'}),kind_risk({'polarity'}),kind_risk({'acronym','entity','high_idf'}),kind_risk({'predicate'}),
        float(max(s['number_alt_conflict'] for s in specs)),float(max(s['acronym_alt_conflict'] for s in specs)),float(max(s['polarity_alt_conflict'] for s in specs)),
        float(1.0-np.mean(present)),float(max(retrieval) if retrieval else 0.0),float(min(retrieval) if retrieval else 0.0)
    ],dtype=np.float64)

def make_feature_frame(indices,bank,support_lookup):
    rows=[]
    for idx in map(int,indices):
        specs=bank[idx]; vec=feature_vector(specs,support_lookup(idx))
        rows.append({'idx':idx,**{k:float(v) for k,v in zip(FEATURE_NAMES,vec)}})
    return pd.DataFrame(rows).set_index('idx')

def new_calibrator(seed):
    return Pipeline([
        ('scale',StandardScaler()),
        ('lr',LogisticRegression(C=CALIBRATOR_C,class_weight=CALIBRATOR_CLASS_WEIGHT,solver='liblinear',max_iter=2000,random_state=seed))
    ])

def choose_guard_threshold(prob_by_idx,fit_mask):
    fit_idx=np.asarray([i for i in train_rescue_idx if fit_mask[i] and i in prob_by_idx],dtype=int)
    if len(fit_idx)==0: return 1.000001,{'selected':0,'precision':np.nan,'preservation':1.0,'gain':0.0}
    probs=np.asarray([prob_by_idx[i] for i in fit_idx],float)
    thresholds=np.r_[1.000001,np.unique(probs)[::-1]]
    base=train_v8a_label[fit_mask]; yy=y[fit_mask]; base_f1=macro_f1(yy,base)
    best=None
    for thr in thresholds:
        sel_idx=fit_idx[probs>=thr]
        selected=len(sel_idx)
        if selected==0:
            precision=np.nan; preservation=1.0; gain=0.0
        else:
            precision=float((y[sel_idx]==0).mean())
            true_total=max(1,int((y[fit_idx]==1).sum()))
            preservation=float(1.0-(y[sel_idx]==1).sum()/true_total)
            pred=train_v8a_label.copy(); pred[sel_idx]=0
            gain=macro_f1(yy,pred[fit_mask])-base_f1
        feasible=(selected>0 and precision>=MIN_REVERT_PRECISION and preservation>=MIN_TRUE_RESCUE_PRESERVATION and gain>0)
        if feasible:
            key=(gain,precision,preservation,-selected,thr)
            if best is None or key>best[0]: best=(key,thr,{'selected':selected,'precision':precision,'preservation':preservation,'gain':gain})
    if best is None: return 1.000001,{'selected':0,'precision':np.nan,'preservation':1.0,'gain':0.0}
    return float(best[1]),best[2]

# Per-repeat OOF feature frames.
train_features=[]
for rep in range(N_REPEATS):
    frame=make_feature_frame(train_rescue_idx,train_span_bank,lambda idx,rep=rep:train_support[rep,idx])
    frame['repeat']=rep; train_features.append(frame)

outer_rows=[]; oof_guard_prob=np.full((N_REPEATS,len(train)),np.nan,float); oof_guard_flag=np.zeros((N_REPEATS,len(train)),bool)
test_fold_prob=np.full((N_REPEATS,N_FOLDS,len(test)),np.nan,float); test_fold_flag=np.zeros((N_REPEATS,N_FOLDS,len(test)),bool)
calibrator_rows=[]

for rep in range(N_REPEATS):
    frame=train_features[rep]
    for fold in range(N_FOLDS):
        fit_idx=np.asarray([i for i in train_rescue_idx if repeat_fold[rep,i]!=fold],int)
        val_idx=np.asarray([i for i in train_rescue_idx if repeat_fold[rep,i]==fold],int)
        Xfit=frame.loc[fit_idx,FEATURE_NAMES].to_numpy(float); tfit=(y[fit_idx]==0).astype(int)
        if np.unique(tfit).size<2: raise RuntimeError(f'repeat={rep} fold={fold}: guard fit target lacks both classes')
        cal=new_calibrator(REPEAT_SEEDS[rep]+fold); cal.fit(Xfit,tfit)
        pfit=cal.predict_proba(Xfit)[:,1]; prob_fit={int(i):float(p) for i,p in zip(fit_idx,pfit)}
        fit_mask=repeat_fold[rep]!=fold; thr,thr_info=choose_guard_threshold(prob_fit,fit_mask)

        Xval=frame.loc[val_idx,FEATURE_NAMES].to_numpy(float); pval=cal.predict_proba(Xval)[:,1]
        flags=pval>=thr
        for i,p,fl in zip(val_idx,pval,flags): oof_guard_prob[rep,i]=float(p); oof_guard_flag[rep,i]=bool(fl)

        # Honest full-fold evaluation.
        hold=repeat_fold[rep]==fold; pred=train_v8a_label.copy(); pred[val_idx[flags]]=0
        base_f1=macro_f1(y[hold],train_v8a_label[hold]); new_f1=macro_f1(y[hold],pred[hold]); gain=new_f1-base_f1
        selected_idx=val_idx[flags]
        rev_prec=float((y[selected_idx]==0).mean()) if len(selected_idx) else np.nan
        true_total=max(1,int((y[val_idx]==1).sum())); preserve=float(1.0-(y[selected_idx]==1).sum()/true_total)
        outer_rows.append({'repeat':rep,'fold':fold,'rows':int(hold.sum()),'v8a_rescues':len(val_idx),'threshold':thr,
                           'fit_selected':thr_info['selected'],'fit_precision':thr_info['precision'],'fit_preservation':thr_info['preservation'],'fit_gain':thr_info['gain'],
                           'heldout_reverts':len(selected_idx),'heldout_revert_precision':rev_prec,'heldout_true_rescue_preservation':preserve,
                           'v8a_macro_f1':base_f1,'v8b_macro_f1':new_f1,'gain':gain})

        # Apply this exact fold guard to test features from the matching fold specialist.
        tframe=make_feature_frame(test_rescue_idx,test_span_bank,lambda idx,rep=rep,fold=fold:test_support[rep,fold,idx])
        pt=cal.predict_proba(tframe.loc[test_rescue_idx,FEATURE_NAMES].to_numpy(float))[:,1]
        for i,p in zip(test_rescue_idx,pt):
            test_fold_prob[rep,fold,i]=float(p); test_fold_flag[rep,fold,i]=bool(p>=thr)

        lr=cal.named_steps['lr']; calibrator_rows.append({'repeat':rep,'fold':fold,'threshold':thr,'intercept':float(lr.intercept_[0]),**{f'coef_{n}':float(v) for n,v in zip(FEATURE_NAMES,lr.coef_[0])}})

outer_metrics=pd.DataFrame(outer_rows)
print('Outer fold metrics:'); display(outer_metrics)

Outer fold metrics:


,repeat,fold,rows,v8a_rescues,threshold,fit_selected,fit_precision,fit_preservation,fit_gain,heldout_reverts,heldout_revert_precision,heldout_true_rescue_preservation,v8a_macro_f1,v8b_macro_f1,gain
0,0,0,4798,44,1.000001,0,NaN,1.0,0.000000,0,NaN,1.000000,0.907126,0.907126,0.000000
1,0,1,4807,25,1.000001,0,NaN,1.0,0.000000,0,NaN,1.000000,0.900270,0.900270,0.000000
2,0,2,4792,39,0.664141,1,1.0,1.0,0.000334,1,0.0,0.971429,0.913421,0.912954,-0.000467
3,1,0,4765,37,0.693018,1,1.0,1.0,0.000320,0,NaN,1.000000,0.908974,0.908974,0.000000
4,1,1,4783,33,1.000001,0,NaN,1.0,0.000000,0,NaN,1.000000,0.899265,0.899265,0.000000
5,1,2,4849,38,0.792536,1,1.0,1.0,0.000327,0,NaN,1.000000,0.913380,0.913380,0.000000


## Aggregate repeated-CV promotion gate and test consensus

A train rescue is reverted only if **both independent repeats** flag it. A test rescue is reverted only if at least **2/3 fold guards flag it inside each repeat**, and both repeat-level majorities agree. This keeps V8B a strict subset of V8A's 18 interventions.

In [9]:
def _counts4(y_true,y_pred):
    y_true=np.asarray(y_true,np.int8); y_pred=np.asarray(y_pred,np.int8)
    return np.array([np.sum((y_true==0)&(y_pred==0)),np.sum((y_true==0)&(y_pred==1)),np.sum((y_true==1)&(y_pred==0)),np.sum((y_true==1)&(y_pred==1))],dtype=np.int64)
def _macro_from_counts(c):
    c=np.asarray(c,dtype=float); tn,fp,fn,tp=np.moveaxis(c,-1,0); d0=2*tn+fp+fn; d1=2*tp+fp+fn
    f0=np.divide(2*tn,d0,out=np.zeros_like(tn,dtype=float),where=d0>0); f1=np.divide(2*tp,d1,out=np.zeros_like(tp,dtype=float),where=d1>0)
    return 0.5*(f0+f1)
def component_bootstrap_delta(y_true,base_pred,new_pred,components,reps=BOOTSTRAP_REPS,seed=BOOTSTRAP_SEED):
    y_true=np.asarray(y_true,np.int8); base_pred=np.asarray(base_pred,np.int8); new_pred=np.asarray(new_pred,np.int8); components=np.asarray(components)
    unique=np.unique(components); bc=np.vstack([_counts4(y_true[components==g],base_pred[components==g]) for g in unique]); nc=np.vstack([_counts4(y_true[components==g],new_pred[components==g]) for g in unique])
    rng=np.random.default_rng(seed); out=[]; batch=100
    for start in range(0,reps,batch):
        n=min(batch,reps-start); draw=rng.integers(0,len(unique),size=(n,len(unique)))
        out.extend((_macro_from_counts(nc[draw].sum(axis=1))-_macro_from_counts(bc[draw].sum(axis=1))).tolist())
    return np.asarray(out,float)

# Aggregate train OOF: two independent repeat decisions must agree.
oof_revert=train_v8a_rescue & oof_guard_flag.all(axis=0)
train_v8b_label=train_v8a_label.copy(); train_v8b_label[oof_revert]=0
assert np.all((~oof_revert)|train_v8a_rescue)
assert not np.any(oof_revert & (train_v4_rule!=REQUIRED_V4_RULE))
agg_base=macro_f1(y,train_v8a_label); agg_new=macro_f1(y,train_v8b_label); agg_gain=agg_new-agg_base
revert_precision=float((y[oof_revert]==0).mean()) if oof_revert.any() else 0.0
true_rescue_total=max(1,int((y[train_v8a_rescue]==1).sum()))
true_preservation=float(1.0-(y[oof_revert]==1).sum()/true_rescue_total)
boot=component_bootstrap_delta(y,train_v8a_label,train_v8b_label,train_component)
boot_positive=float((boot>0).mean()); ci_low,ci_high=np.quantile(boot,[0.025,0.975])
nonneg_folds=int((outer_metrics.gain>=-1e-12).sum()); positive_folds=int((outer_metrics.gain>1e-12).sum())

VALIDATION_OK=bool(
    agg_gain>MIN_AGG_GAIN and revert_precision>=MIN_REVERT_PRECISION and true_preservation>=MIN_TRUE_RESCUE_PRESERVATION
    and nonneg_folds>=MIN_NONNEGATIVE_OUTER_FOLDS and positive_folds>=MIN_POSITIVE_OUTER_FOLDS and boot_positive>=MIN_BOOTSTRAP_POSITIVE
)

aggregate_metrics=pd.DataFrame([{
    'v8a_macro_f1':agg_base,'v8b_macro_f1':agg_new,'gain':agg_gain,'oof_reverts':int(oof_revert.sum()),'revert_precision':revert_precision,
    'true_rescue_preservation':true_preservation,'nonnegative_outer_folds':nonneg_folds,'positive_outer_folds':positive_folds,
    'bootstrap_median_gain':float(np.median(boot)),'bootstrap_ci_low_95':float(ci_low),'bootstrap_ci_high_95':float(ci_high),
    'bootstrap_positive_fraction':boot_positive,'validation_ok':VALIDATION_OK
}])
print('Aggregate validation:'); display(aggregate_metrics)

# Conservative test consensus: majority within each repeat, agreement across repeats.
repeat_test_flag=np.zeros((N_REPEATS,len(test)),bool)
for rep in range(N_REPEATS): repeat_test_flag[rep]=test_v8a_rescue & (test_fold_flag[rep].sum(axis=0)>=2)
test_revert=test_v8a_rescue & repeat_test_flag.all(axis=0)
test_revert_rate=float(test_revert.sum()/max(1,test_v8a_rescue.sum()))
coverage_ok=bool(test_revert_rate<=MAX_OBVIOUS_TEST_REVERT_FRACTION)
nontrivial=bool(test_revert.sum()>=MIN_TEST_REVERTS)

# V8B may only restore V5 on a subset of V8A rescues.
proposed_test_v8b=test_v8a_label.copy(); proposed_test_v8b[test_revert]=0
assert np.all((~test_revert)|test_v8a_rescue)
assert np.all(proposed_test_v8b[~test_v8a_rescue]==test_v5_label[~test_v8a_rescue])
assert np.all(proposed_test_v8b[test_revert]==test_v5_label[test_revert])
assert not np.any((test_v5_label==1)&(proposed_test_v8b==0))
assert int(np.sum(proposed_test_v8b!=test_v5_label))<=EXPECTED_V8A_TEST_RESCUES

PROMOTED=bool(VALIDATION_OK and coverage_ok and nontrivial)
print('Test V8A proposals:',int(test_v8a_rescue.sum()),'| V8B guard reverts:',int(test_revert.sum()),'| promoted:',PROMOTED)

Aggregate validation:


,v8a_macro_f1,v8b_macro_f1,gain,oof_reverts,revert_precision,true_rescue_preservation,nonnegative_outer_folds,positive_outer_folds,bootstrap_median_gain,bootstrap_ci_low_95,bootstrap_ci_high_95,bootstrap_positive_fraction,validation_ok
0,0.907147,0.907147,0.0,0,0.0,1.0,5,0,0.0,0.0,0.0,0.0,False


Test V8A proposals: 18 | V8B guard reverts: 0 | promoted: False


## Export complete audit artifacts and submission

If the promotion gate fails, the notebook writes a clearly named frozen-V5 fallback and **does not** create `submission_v8b.csv`. This prevents accidental submission of an unvalidated guardrail.

In [10]:
# Candidate-level train diagnostics.
train_candidate_rows=[]
for idx in train_rescue_idx:
    row={'idx':int(idx),'id':train.iloc[idx].id,'label':int(y[idx]),'v5_label':int(train_v5_label[idx]),'v8a_label':int(train_v8a_label[idx]),'v3_blend_prob':float(train_v3_blend[idx]),
         'repeat0_prob':float(oof_guard_prob[0,idx]),'repeat1_prob':float(oof_guard_prob[1,idx]),'repeat0_flag':int(oof_guard_flag[0,idx]),'repeat1_flag':int(oof_guard_flag[1,idx]),'v8b_revert':int(oof_revert[idx])}
    for rep in range(N_REPEATS):
        fv=feature_vector(train_span_bank[int(idx)],train_support[rep,idx]); row.update({f'r{rep}_{k}':float(v) for k,v in zip(FEATURE_NAMES,fv)})
    train_candidate_rows.append(row)
train_candidate_scores=pd.DataFrame(train_candidate_rows)

# Test candidate diagnostics from all six fold guards.
test_candidate_rows=[]
for idx in test_rescue_idx:
    row={'idx':int(idx),'id':test.iloc[idx].id,'title':test.iloc[idx].title,'v5_label':int(test_v5_label[idx]),'v8a_label':int(test_v8a_label[idx]),'v3_blend_prob':float(test_v3_blend[idx]),
         'repeat0_majority':int(repeat_test_flag[0,idx]),'repeat1_majority':int(repeat_test_flag[1,idx]),'v8b_revert':int(test_revert[idx]),'proposed_v8b_label':int(proposed_test_v8b[idx])}
    for rep in range(N_REPEATS):
        for fold in range(N_FOLDS): row[f'r{rep}f{fold}_prob']=float(test_fold_prob[rep,fold,idx]); row[f'r{rep}f{fold}_flag']=int(test_fold_flag[rep,fold,idx])
    test_candidate_rows.append(row)
test_candidate_scores=pd.DataFrame(test_candidate_rows)
changed_rows=test_candidate_scores[test_candidate_scores.v8b_revert==1].merge(test[['id','title','content']],on=['id','title'],how='left')

pair_df=pd.DataFrame(pair_metric_rows)
pair_summary=(pair_df.groupby(['repeat','fold']).agg(pair_count=('correct','size'),pair_ranking_accuracy=('correct','mean')).reset_index() if len(pair_df) else pd.DataFrame())
split_registry=pd.DataFrame({'id':train.id,'component':train_component,'repeat0_fold':repeat_fold[0],'repeat1_fold':repeat_fold[1]})
bootstrap_df=pd.DataFrame({'replicate':np.arange(BOOTSTRAP_REPS),'delta_macro_f1':boot})
calibrator_df=pd.DataFrame(calibrator_rows)
training_df=pd.DataFrame(training_history)

artifacts={
    'split_registry_v8b.csv':split_registry,
    'counterfactual_pairs_v8b.csv':clean_pairs,
    'training_history_v8b.csv':training_df,
    'pair_metrics_v8b.csv':pair_df,
    'pair_summary_v8b.csv':pair_summary,
    'calibrator_folds_v8b.csv':calibrator_df,
    'outer_fold_metrics_v8b.csv':outer_metrics,
    'aggregate_metrics_v8b.csv':aggregate_metrics,
    'bootstrap_v8b.csv':bootstrap_df,
    'train_candidate_scores_v8b.csv':train_candidate_scores,
    'test_candidate_scores_v8b.csv':test_candidate_scores,
    'changed_rows_v8b.csv':changed_rows,
}
for name,df in artifacts.items(): df.to_csv(OUTPUT_DIR/name,index=False)

if PROMOTED:
    submission=sample[['id']].copy(); submission['label']=proposed_test_v8b.astype(int)
    assert submission.id.astype(str).tolist()==test.id.astype(str).tolist(); assert set(submission.label.unique()).issubset({0,1})
    submission.to_csv(OUTPUT_DIR/'submission_v8b.csv',index=False); submission.to_csv(OUTPUT_DIR/'submission.csv',index=False)
    print('SUBMIT artifact created:',OUTPUT_DIR/'submission_v8b.csv')
else:
    fallback=sample[['id']].copy(); fallback['label']=test_v5_label.astype(int)
    fallback.to_csv(OUTPUT_DIR/'submission_v5_fallback_DO_NOT_SUBMIT_AS_V8B.csv',index=False)
    print('V8B failed its frozen promotion contract. No submission_v8b.csv was created.')

config={
    'version':'v8b-guarded-semantic-rescue','base':'exact-frozen-v5','proposer':'exact-frozen-v8a','direction':'revert-subset-of-v8a-only',
    'v8a_threshold':V8_BLEND_MIN,'model_name':MODEL_NAME,'model_revision':MODEL_REVISION,'repeat_seeds':list(REPEAT_SEEDS),'folds':N_FOLDS,
    'training':{'epochs':EPOCHS,'lr':LR,'max_length':MAX_LENGTH,'window_words':SPAN_WINDOW_WORDS,'window_stride':SPAN_WINDOW_STRIDE,'top_windows':SPAN_TOP_WINDOWS,'max_spans':MAX_SPANS,
                'row_batch_size':ROW_BATCH_SIZE,'grad_accum':GRAD_ACCUM,'strong_loss_weight':STRONG_LOSS_WEIGHT,'pair_loss_weight':PAIR_LOSS_WEIGHT,'row_mil_weight':ROW_MIL_WEIGHT},
    'calibrator':{'type':'StandardScaler+LogisticRegression','C':CALIBRATOR_C,'class_weight':CALIBRATOR_CLASS_WEIGHT,'features':FEATURE_NAMES,
                  'min_revert_precision':MIN_REVERT_PRECISION,'min_true_rescue_preservation':MIN_TRUE_RESCUE_PRESERVATION},
    'promotion':{'min_gain':MIN_AGG_GAIN,'min_nonnegative_outer_folds':MIN_NONNEGATIVE_OUTER_FOLDS,'min_positive_outer_folds':MIN_POSITIVE_OUTER_FOLDS,'min_bootstrap_positive':MIN_BOOTSTRAP_POSITIVE,
                 'max_test_revert_fraction':MAX_OBVIOUS_TEST_REVERT_FRACTION,'min_test_reverts':MIN_TEST_REVERTS},
    'status':{'validation_ok':VALIDATION_OK,'coverage_ok':coverage_ok,'nontrivial':nontrivial,'promoted':PROMOTED},
    'test':{'v8a_proposals':int(test_v8a_rescue.sum()),'guard_reverts':int(test_revert.sum()),'final_changes_vs_v5':int((proposed_test_v8b!=test_v5_label).sum())},
}
provenance={'train_path':str(train_path),'test_path':str(test_path),'sample_path':str(sample_path),'train_id_sha256':ids_sha256(train.id),'test_id_sha256':ids_sha256(test.id),
            'frozen_artifact_sha256':_FROZEN_SHA256,'frozen_v5_submission_sha256':v5_sha,'model_sources':sorted(set(map(str,model_sources))),'timestamp_utc':pd.Timestamp.utcnow().isoformat()}
(OUTPUT_DIR/'config_v8b.json').write_text(json.dumps(config,indent=2,default=str))
(OUTPUT_DIR/'provenance_v8b.json').write_text(json.dumps(provenance,indent=2,default=str))

manifest={}
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file() and p.name not in {'manifest_v8b.json','results_v8b.zip'}:
        manifest[p.name]={'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
(OUTPUT_DIR/'manifest_v8b.json').write_text(json.dumps(manifest,indent=2))
with zipfile.ZipFile(OUTPUT_DIR/'results_v8b.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file() and p.name!='results_v8b.zip': z.write(p,arcname=p.name)
print('Results bundle:',OUTPUT_DIR/'results_v8b.zip')
print('Final status:',config['status'])

V8B failed its frozen promotion contract. No submission_v8b.csv was created.
Results bundle: /kaggle/working/ifest_v8b_output/results_v8b.zip
Final status: {'validation_ok': False, 'coverage_ok': True, 'nontrivial': False, 'promoted': False}
